# Default Estimator and WoE Binning

## Construct RFMS Features

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../assets/datas/data.csv')

# Convert TransactionStartTime to datetime
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Recency: Time since the last transaction
df['Recency'] = (df['TransactionStartTime'].max() - df['TransactionStartTime']).dt.days

# Frequency: Number of transactions per customer
frequency = df.groupby('CustomerId')['TransactionId'].count().reset_index()
frequency.columns = ['CustomerId', 'Frequency']

# Monetary: Total amount spent per customer
monetary = df.groupby('CustomerId')['Amount'].sum().reset_index()
monetary.columns = ['CustomerId', 'Monetary']

# Seasonality: Standard deviation of transaction amounts per customer
seasonality = df.groupby('CustomerId')['Amount'].std().reset_index()
seasonality.columns = ['CustomerId', 'Seasonality']

# Merge RFMS features
rfms = pd.merge(frequency, monetary, on='CustomerId')
rfms = pd.merge(rfms, seasonality, on='CustomerId')
rfms = pd.merge(rfms, df[['CustomerId', 'Recency']].drop_duplicates(), on='CustomerId')

print(rfms.head())

        CustomerId  Frequency  Monetary  Seasonality  Recency
0     CustomerId_1          1  -10000.0          NaN       83
1    CustomerId_10          1  -10000.0          NaN       83
2  CustomerId_1001          5   20000.0  6558.963333       89
3  CustomerId_1002         11    4225.0   560.498966       89
4  CustomerId_1002         11    4225.0   560.498966       82
